# 2.4 Agent Executors, Errors & Retries

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Explain why the old `AgentExecutor` is legacy and which 1.x mechanisms replaced its error handling, retries, and stopping conditions.
- Choose the right error strategy per tool: return errors as data, convert exceptions with `ToolErrorMiddleware`, or retry transient failures with `ToolRetryMiddleware`.
- Cap an agent's cost with `ModelCallLimitMiddleware` and `ToolCallLimitMiddleware`, and know how those differ from the `recursion_limit` backstop.
- Add model-level resilience with `.with_retry()` and `.with_fallbacks()` on a chat model.
- Make a hand-rolled ReAct loop resilient: step budget, retry on model calls, errors as `ToolMessage`s, and a hard backstop.

## Where AgentExecutor went

In pre-1.0 LangChain, `AgentExecutor` was the harness: it ran the thought/action/observation loop, caught parsing errors, retried failed tool calls, and stopped after `max_iterations`. In LangChain 1.x that single class was decomposed into pieces with sharper jobs:

- The loop itself is the LangGraph runtime inside `create_agent`.
- Parsing errors barely exist any more, because current models emit native tool calls instead of text the harness must parse.
- Retries and limits moved into **middleware**, configured per concern instead of per agent.
- Stopping conditions split into two: a semantic budget (middleware) and an infrastructure backstop (`recursion_limit`).

`AgentExecutor` still runs from the `langchain-classic` maintenance package, but it is frozen there. Do not build on it. Everything in this lesson is the current replacement.

## Errors as data: the cheapest fix

Most tool failures are not code bugs. They are bad arguments, missing records, or flaky services, and the model's next thought can usually recover from them if it can *see* the failure. That gives the first and cheapest pattern: return the error as a normal string.

In [ ]:
from langchain_core.tools import tool

@tool
def divide(a: float, b: float) -> str:
    """Divide a by b. Returns an error string for bad input instead of crashing."""
    if b == 0:
        return "Error: cannot divide by zero. Ask the user for a non-zero divisor."
    return str(a / b)

The tool never raises, the loop never breaks, and the next thought reads the error like any other observation. Lesson 2.2 taught this as friendly errors; here is why it matters at the harness level: it converts a whole class of failures from crashes into recoverable observations, with no middleware at all. Prefer it for deterministic failures (bad input, record not found, validation errors) where retrying the identical call would just burn tokens.

## ToolErrorMiddleware: converting exceptions

Some tools must raise. A database client raises on connection failure; an HTTP client raises on timeouts. When a tool raises inside `create_agent`, the run halts by default. `ToolErrorMiddleware` is the 1.x mechanism that catches exceptions per tool and converts them into error `ToolMessage`s the model can read and recover from.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[lookup_order, divide],
    system_prompt="You are a support agent. If a tool reports an error, read it and try a corrected call.",
    middleware=[
        ToolErrorMiddleware(
            on_error=lambda exc, request: (
                f"Tool {request['tool_call']['name']} failed: {exc}. "
                "Check your arguments and try again, or tell the user what went wrong."
            )
        )
    ],
)

Notes worth keeping:

- `on_error` receives the exception and the tool-call request and returns a string, or returns `None` to let the exception propagate and halt the run. Returning `None` is how you say "this failure is not recoverable".
- Scope it with `tools=[...]` when only some tools should be caught. A payment tool's exceptions should probably propagate loudly; a search tool's should become observations.
- The `@tool(handle_tool_error=...)` parameter still exists (it accepts a bool, a string, or a callable) and works per tool. The middleware is the documented primary pattern in 1.x because one rule covers many tools and keeps the policy visible at the agent level instead of scattered across tool definitions.

## ToolRetryMiddleware: retry only what is transient

Converting an error into an observation lets the model *correct itself*. Retrying re-runs the *identical* call, which only helps when the failure is transient: timeouts, connection resets, rate limits. Retrying a deterministic failure (record not found, validation error) is just burning money, so `ToolRetryMiddleware` takes a `retry_on` filter:

In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

middleware = [
    ToolRetryMiddleware(
        max_retries=2,                       # retries after the first call: 3 attempts total
        retry_on=(TimeoutError, ConnectionError),
        backoff_factor=2.0,
        initial_delay=1.0,
        max_delay=60.0,
        jitter=True,
        on_failure="continue",               # after retries: send the error to the model
    ),
]

- `on_failure="continue"` (the default) sends an error `ToolMessage` after the retries are exhausted, so the model gets one last chance to recover. `"error"` raises instead, and a callable lets you decide per failure.
- Scope with `tools=[...]` when only some tools are worth retrying.
- Exponential backoff with jitter is the default for a reason: a fleet of agents retrying in lockstep turns a provider hiccup into a self-inflicted DDoS. Do not set `jitter=False` unless you know why.

The rule to carry: **correct deterministic failures, retry transient ones.** "Error as data" for the first, `ToolRetryMiddleware` for the second.

## Budgets: call limits and the recursion backstop

Two different mechanisms cap how much an agent may do, and you want both.

**Middleware budgets** count model calls or tool calls and act semantically:

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

middleware = [
    ModelCallLimitMiddleware(run_limit=25),          # max 25 model calls per run
    ToolCallLimitMiddleware(                         # search is the expensive tool: cap it
        tool_name="web_search",
        run_limit=5,
        exit_behavior="continue",                    # block extra calls with an error message
    ),
]

- `ModelCallLimitMiddleware(run_limit=...)` caps model calls per single invocation; `thread_limit=...` caps across a whole thread (needs a checkpointer, Module 4). `exit_behavior="end"` stops gracefully, `"error"` raises `ModelCallLimitExceededError`.
- `ToolCallLimitMiddleware` with `tool_name` omitted caps all tools globally. Its `"continue"` mode (the default) blocks the excess call and feeds an error `ToolMessage` to the model, so the agent can wrap up with what it has.

**`recursion_limit`** is the infrastructure backstop. It caps LangGraph super-steps, and one ReAct iteration costs roughly two steps (one model call, one tool execution), so `recursion_limit=20` allows about 9 iterations. Set it on invoke:

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Research the EU AI Act timeline."}]},
        config={"recursion_limit": 20},
    )
except GraphRecursionError:
    print("Hit the step ceiling. Inspect result['messages'] to find the loop.")

Hitting it raises `GraphRecursionError`; it does not degrade gracefully. That is the point: it is the last line of defence against a bug the middleware budget missed. Use middleware for cost control and user experience, `recursion_limit` as the hard backstop. The LangGraph default is 25.

## Model-level resilience: retries and fallbacks

Tool failures are only half the story. Model calls fail too: rate limits, provider outages, malformed responses. Every `Runnable` in langchain-core has `.with_retry()`:

In [ ]:
from langchain_openai import ChatOpenAI
import openai

primary = ChatOpenAI(model="gpt-4o-mini")
resilient = primary.with_retry(
    retry_if_exception_type=(openai.RateLimitError, openai.APIConnectionError),
    stop_after_attempt=3,
)

And `.with_fallbacks()` tries a second model when the first keeps failing:

In [ ]:
model = primary.with_retry(stop_after_attempt=2).with_fallbacks(
    [ChatOpenAI(model="gpt-4o-mini", temperature=0.0)]
)

Inside `create_agent`, the 1.x-idiomatic equivalents are middleware: `ModelRetryMiddleware` (same backoff parameters as the tool version, applied to model calls) and `ModelFallbackMiddleware` (takes model id strings like `"openai:gpt-4o-mini"` or model instances). The LCEL primitives and the middleware do the same job; use the middleware when you are building with `create_agent` so the policy sits with the agent, and the LCEL versions when you are composing chains by hand.

One caution: fallbacks are not free redundancy. A fallback model should be cheaper or more available than the primary, and switching models mid-conversation changes behaviour subtly (different tool-call habits, different tone). Log when a fallback fires; Module 6 shows you how.

## Making a hand-rolled loop resilient

Lesson 2.3's hand-rolled loop is a teaching tool, but sometimes you ship one (non-standard control flow, custom stop conditions). A production hand-rolled loop needs four things:

In [ ]:
import hashlib
from langchain_core.messages import ToolMessage
from langchain_core.tools import ToolException

MAX_STEPS = 8
seen_calls = set()
model = ChatOpenAI(model="gpt-4o-mini").with_retry(stop_after_attempt=3)  # 1. retry model calls

for step in range(MAX_STEPS):                                            # 2. step budget
    ai_msg = model.invoke(messages)
    messages.append(ai_msg)
    if not ai_msg.tool_calls:
        break
    for call in ai_msg.tool_calls:
        fingerprint = hashlib.sha256(
            f"{call['name']}:{sorted(call['args'].items())}".encode()
        ).hexdigest()
        if fingerprint in seen_calls:                                    # 3. kill repeat loops
            messages.append(ToolMessage(
                content="You already called this tool with these exact arguments. "
                        "Do not call it again; use what you have or ask the user.",
                tool_call_id=call["id"],
            ))
            continue
        seen_calls.add(fingerprint)
        try:
            messages.append(by_name[call["name"]].invoke(call))
        except ToolException as e:                                       # 4. errors as data
            messages.append(ToolMessage(content=f"Error: {e}", tool_call_id=call["id"], status="error"))

Even then, invoke with `config={"recursion_limit": 25}` as the backstop, and note what is still missing versus `create_agent`: no streaming, no persistence, no human-in-the-loop. Hand-rolling is for control flow the harness cannot express, not for reimplementing its reliability.

## Timeouts: do them inside the tool

No middleware in the pinned stack sets a per-tool timeout. A tool that hangs on a network call hangs the whole agent. Put the timeout where the wait happens:

In [ ]:
import httpx
from langchain_core.tools import tool

@tool
def fetch_status(url: str) -> str:
    """Fetch the HTTP status of a URL. Times out after 10 seconds."""
    try:
        r = httpx.get(url, timeout=10.0)
        return f"{url}: HTTP {r.status_code}"
    except httpx.TimeoutException:
        return f"Error: {url} did not respond within 10 seconds."

Mark timeouts as retryable in `retry_on` (they usually are transient), or return them as error strings if the model should move on. Either way, a tool without a timeout is a tool that can hang forever.

## What can go wrong

- **Infinite loops are a billing event.** A model that misreads an observation and re-issues the same call pays full price per iteration, with the history growing each time. Hard ceiling (`recursion_limit`), semantic budget (`ModelCallLimitMiddleware`), and duplicate-call detection: all three, not one.
- **Retrying deterministic failures.** `retry_on` with a bare `Exception` retries validation errors and missing records that will never succeed. Filter to transient errors; convert the rest to observations the model can learn from.
- **Silent swallowing.** A `handle_tool_error` string or a broad `on_error` can hide real bugs (a broken credential, a renamed API field) behind polite messages the model "recovers" from by hallucinating. Log every converted error server-side; the model sees the friendly version, your logs see the traceback.
- **Side-effecting tools in retry loops.** A tool that sends an email or charges a card can execute twice: once before the timeout, once on retry. Retry only idempotent tools; gate side effects behind confirmation arguments or human-in-the-loop middleware.
- **Fallback behaviour drift.** A fallback model is a different model. Its tool-call style and tone differ, which can confuse multi-turn conversations. Log fallback activations and test the degraded path, not just the happy one.
- **Error text leaking into logs.** Tool errors often contain user data, file paths, or credentials. If you persist traces (Module 6), treat error messages as potentially sensitive: redact before storing, especially under GDPR.
- **Timeout without retry policy.** A 10-second tool timeout with no `retry_on` turns every transient blip into a user-visible failure; a timeout with unbounded retries turns one blip into a 5-minute hang. Set both together.

## Check your understanding

**1. Name the 1.x replacements for the three jobs the old AgentExecutor did: the loop, error handling, and stopping conditions.**
The loop is the LangGraph runtime inside `create_agent`. Error handling is `ToolErrorMiddleware` (plus errors-as-data in the tools). Stopping conditions split into semantic budgets (`ModelCallLimitMiddleware` / `ToolCallLimitMiddleware`) and the infrastructure backstop (`recursion_limit` on invoke).

**2. A tool raises `ConnectionError` on flaky networks and `ValueError("order not found")` on bad ids. Configure the agent's error handling.**
Return the `ValueError` path as an error string (deterministic: the model corrects the id), and add `ToolRetryMiddleware(retry_on=(ConnectionError,), max_retries=2)` for the transient network failure. Optionally add `ToolErrorMiddleware` as a catch-all so unexpected exceptions become observations instead of crashes.

**3. What is the difference between `ModelCallLimitMiddleware(run_limit=25)` and `config={"recursion_limit": 50}`?**
The middleware counts model calls and stops the agent gracefully (`"end"`) or with a named error (`"error"`); it is a cost/UX budget. `recursion_limit` counts LangGraph super-steps (about two per ReAct iteration) and raises `GraphRecursionError` when exceeded; it is the hard infrastructure backstop. Use both: middleware for control, `recursion_limit` for safety.

**4. Why must `retry_on` be narrow, and what goes wrong with `retry_on=(Exception,)`?**
A broad filter retries deterministic failures (bad arguments, missing records, validation errors) that will never succeed, burning a model call and a tool call per attempt. Narrow it to transient errors (timeouts, connection resets, rate limits) so retries only fire where a second attempt can plausibly work.

**5. Your hand-rolled agent keeps calling `lookup_order(order_id="42")` forever. Name the two defences this lesson adds and where each lives.**
Duplicate-call detection in the loop (hash tool name plus arguments, refuse repeats with a `ToolMessage` telling the model to move on), and the `recursion_limit` backstop on invoke, which raises `GraphRecursionError` when the step ceiling is hit. The first prevents the waste; the second guarantees it cannot run forever.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.